## ARC-AGI-3 Agent Manual Playthrough

ARC-AGI-3 is an Interactive Reasoning Benchmark that tests an *agents* ability to reason, remember, and orient goals.

This notebook is for educational purposes. We will build a random agent which will help us review what the [swarm](http://docs.arcprize.org/swarms) does automatically. This should give you a better feel about what is happening under the hood.

To view more, see the [docs](docs.arcprize.org).

First up, let's set up our environment

In [ ]:
#!/usr/bin/env python3
"""
Simple demo showing what a swarm agent does under the hood.
This is a bare-bones example for educational purposes.
"""

import json
import os
import random
import requests

# Setup
ROOT_URL = "https://three.arcprize.org"
API_KEY = '<your ARC-AGI-API-KEY>' # Found at three.arcprize.org

# Create a requests session with headers
session = requests.Session()
session.headers.update({
    "X-API-Key": API_KEY,
    "Accept": "application/json"
})

Next, we are going get a random game. To do this we'll first call /api/games to get a list, then we'll pick randomly.

In [ ]:
print("=== MANUAL SWARM DEMO ===")
print("This shows what happens when an agent plays an ARC game.\n")

# Step 1: Get available games
print("STEP 1: Getting list of games...")
response = session.get(f"{ROOT_URL}/api/games")
games = [g["game_id"] for g in response.json()]
print(f"Found {len(games)} games")

# Pick a random game
game_id = random.choice(games)
print(f"Selected game: {game_id}\n")

=== MANUAL SWARM DEMO ===
This shows what happens when an agent plays an ARC game.

STEP 1: Getting list of games...
Found 3 games
Selected game: ls20-016295f7601e



Then we need to open a scorecard. Every playthrough must have a scorecard attached to it. A scorecard is the way you'll keep track of your progress and aggregate scores.

In [ ]:
# Step 2: Open a scorecard (tracks performance)
print("STEP 2: Opening scorecard...")
response = session.post(
    f"{ROOT_URL}/api/scorecard/open",
    json={"tags": ["manual_demo"]}
)
card_id = response.json()["card_id"]
print(f"Scorecard ID: {card_id}\n")

STEP 2: Opening scorecard...
Scorecard ID: d1076baf-e0a6-473b-9da9-79a6a896a592



Each game starts with a reset action. This ensures you're ready to go.

In [ ]:
# Step 3: Start the game
print("STEP 3: Starting game with RESET action...")
url = f"{ROOT_URL}/api/cmd/RESET"
print(f"URL: {url}")
response = session.post(
    url,
    json={
        "game_id": game_id,
        "card_id": card_id
    }
)

STEP 3: Starting game with RESET action...
URL: https://three.arcprize.org/api/cmd/RESET


Let's check to make sure everything is set up correctly.

In [ ]:
# Check if response is valid
if response.status_code != 200:
    print(f"Error: {response.status_code} - {response.text}")
    exit()

game_data = response.json()
guid = game_data["guid"]
state = game_data["state"]
score = game_data.get("score", 0)
print(f"Game started! State: {state}, Score: {score}\n")

Game started! State: NOT_FINISHED, Score: 0



Now let's actually take some actions. For simplicity sake, we'll pick random actions. This is where you'll create magic for your agent to try and beat the game.

In [ ]:
# Step 4: Play with random actions (max 5 actions)
print("STEP 4: Taking random actions...")
actions = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION6"]

for i in range(5):
    # Check if game is over
    if state in ["WIN", "GAME_OVER"]:
        print(f"\nGame ended! Final state: {state}, Score: {score}")
        break

    # Pick a random action
    action = random.choice(actions)

    # Build request data
    request_data = {
        "game_id": game_id,
        "card_id": card_id,
        "guid": guid # This is your session ID
    }

    # ACTION6 needs x,y coordinates
    if action == "ACTION6":
        request_data["x"] = random.randint(0, 29)
        request_data["y"] = random.randint(0, 29)
        print(f"Action {i+1}: {action} at ({request_data['x']}, {request_data['y']})", end="")
    else:
        print(f"Action {i+1}: {action}", end="")

    # Take the action
    response = session.post(
        f"{ROOT_URL}/api/cmd/{action}",
        json=request_data
    )

    game_data = response.json()
    state = game_data["state"]
    score = game_data.get("score", 0)
    print(f" -> State: {state}, Score: {score}")

STEP 4: Taking random actions...
Action 1: ACTION2 -> State: NOT_FINISHED, Score: 0
Action 2: ACTION2 -> State: NOT_FINISHED, Score: 0
Action 3: ACTION2 -> State: NOT_FINISHED, Score: 0
Action 4: ACTION1 -> State: NOT_FINISHED, Score: 0
Action 5: ACTION1 -> State: NOT_FINISHED, Score: 0


Now that we went through 5 random actions let's close up the scorecard. This tells the scorecard you're done playing and to finalize.

In [ ]:
# Step 5: Close scorecard
print("\nSTEP 5: Closing scorecard...")
response = session.post(
    f"{ROOT_URL}/api/scorecard/close",
    json={"card_id": card_id}
)
scorecard = response.json()
print("Scorecard closed!")
print(f"\nView results at: {ROOT_URL}/scorecards/{card_id}")

print("\n=== DEMO COMPLETE ===")
print("\nThis is what every agent does:")
print("1. Get games list")
print("2. Open a scorecard")
print("3. Reset to start the game")
print("4. Take actions based on its strategy (we used random)")
print("5. Close the scorecard when done")
print("\nThe real agents use smarter strategies instead of random!")


STEP 5: Closing scorecard...
Scorecard closed!

View results at: https://three.arcprize.org/scorecards/d1076baf-e0a6-473b-9da9-79a6a896a592

=== DEMO COMPLETE ===

This is what every agent does:
1. Get games list
2. Open a scorecard
3. Reset to start the game
4. Take actions based on its strategy (we used random)
5. Close the scorecard when done

The real agents use smarter strategies instead of random!
